In [1]:
import json
from models.swissai import SwissAI


class DotDict(dict):
    def __getattr__(self, key):
        try:
            return self[key]
        except KeyError:
            raise AttributeError(f"'DotDict' object has no attribute '{key}'")

    def __setattr__(self, key, value):
        self[key] = value
        
        
with open("/nfs/scistore19/alistgrp/apanfero/models/apertus3-1b-21n-cooldown-900k/config.json", "r") as f:
    transformers_config = json.load(f)

mapping = {
  "hidden_size": "n_embd",
  "num_attention_heads": "n_head",
  "num_hidden_layers": "n_layer",
  "attention_dropout": "dropout",
  "attention_bias": "bias",
  "initializer_range": "init_std",
  "rms_norm_eps": "rmsnorm_eps",
  "intermediate_size": "intermediate_size",
  "max_position_embeddings": "sequence_length",
}

transformers_config = {
  mapping.get(k, k): v for k, v in transformers_config.items()
}

config = DotDict({
  "vocab_size": 131072,
  "w_quant": "NoQuantizer",
  "w_quant_kwargs": {},
  "a_quant": "NoQuantizer",
  "a_quant_kwargs": {},
  "parallel_block": False,
} | transformers_config)


model = SwissAI(config)

In [2]:
# LOAD /nfs/scistore19/alistgrp/apanfero/models/apertus3-1b-21n-cooldown-900k/model.safetensors"

from safetensors.torch import load_file

model_path = "/nfs/scistore19/alistgrp/apanfero/models/apertus3-1b-21n-cooldown-900k/model.safetensors"
state_dict = load_file(model_path)

def key_mapping(key):
    match key:
        case "model.layers.0.attention_layernorm.weight":
            return "transformer.h.0.attention_layernorm.weight"
        case "model.layers.0.feedforward_layernorm.weight":
            return "transformer.h.0.feedforward_layernorm.weight"
        case "lm_head.weight":
            return "lm_head.weight"
        case "model.embed_tokens.weight":
            return "transformer.wte.weight"
        case "model.norm.weight":
            return "transformer.ln_f.weight"
        case block:
            return f"transformer.h.{block[13:]}"

# Create new state dict with mapped keys
new_state_dict = {}
for k, v in state_dict.items():
    new_state_dict[key_mapping(k)] = v

# Set strict=False to ignore missing keys
model.load_state_dict(new_state_dict, strict=True)

<All keys matched successfully>

In [3]:
model = model.to("cuda")

In [4]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("/nfs/scistore19/alistgrp/apanfero/models/apertus3-1b-21n-cooldown-900k")

In [15]:
import torch

def generate_text_greedily(model, tokenizer, prompt, max_length=50, device='cuda'):
    model.eval()
    input_ids = tokenizer.encode(prompt, return_tensors='pt').to(device)
    
    for _ in range(max_length):
        with torch.no_grad():
            outputs = model(input_ids, get_logits=True)
            logits = outputs['logits'][:, -1, :]
        
        next_token_id = torch.argmax(logits, dim=-1).unsqueeze(-1)
        input_ids = torch.cat([input_ids, next_token_id], dim=-1)
        
    return tokenizer.decode(input_ids[0], skip_special_tokens=True)

generated_text = generate_text_greedily(model, tokenizer, "Swiss AI", max_length=20)
print(generated_text)


Swiss AI and the world’s largest AI company, is a leading AI company, is a leading AI company,
